# Predicting Protein Prediction

Consider as starting point for this exercise a UCI Protein Structure. The dataset comes from the Critical Assessment of protein Structure Prediction experiments (CASP), which is a recurrent (biannual) initiative to predict protein structure from experimental data.

The dataset consists of roughly 45k entries with nine features and one target. 

The features essentially are calculated physicochemical descriptors:
- F1: Total surface area (Approximate exposed surface of the protein)
- F2: Non-polar exposed area (Hydrophobic surface)
- F3: Fraction of exposed nonpolar area (Ratio of hydrophobic and total surface)
- F4: Residue surface exposure (How much amino acids are exposed)
- F5: Secondary structure agreement (Measures consistency with expected structures (α-helices, β-sheets))
- F6: Pairwise distance features (Encodes distances between residues)
- F7: Compactness / packing (How tightly folded the protein is)
- F8: Structural energy-related feature (Proxy for physical plausibility)
- F9: Additional geometric descriptor (Captures global structure properties)

The target is the RMSD (Root Mean Squared Deviation) that describes the deviation of the predicted from the true protein structure. 

The aim of the exercise is to build a model to predict how accurate predicted structures would be based on calculated descriptors.

#### Tasks:
1) The data is somewhat abstract. Inspect it to see what can be expected of a potential model.
2) Create feature matrix and target vector.
3) Choose one Regression ML model, build it and optimise (consider scaling if the model class needs it)
4) Take note of the training and test time for your model (approximation is enough)
5) Whatever model you end up using, try to optimise for accuracy and minimal overfitting, use **MSE** for evaluating your model!
6) Respond to the discussion points.

#### Note:
Feel free in your choice in model class, everything covered in the course so far is on the table. You don't need to compare different ones, we will do that with the compiled results of all assignments.

In [ ]:
# complete imports if needed for your solution
import pandas as pd
import numpy as np


Load and investigate the data

In [ ]:
df = pd.read_csv("CASP.csv")
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45730 entries, 0 to 45729
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   RMSD    45730 non-null  float64
 1   F1      45730 non-null  float64
 2   F2      45730 non-null  float64
 3   F3      45730 non-null  float64
 4   F4      45730 non-null  float64
 5   F5      45730 non-null  float64
 6   F6      45730 non-null  float64
 7   F7      45730 non-null  float64
 8   F8      45730 non-null  int64  
 9   F9      45730 non-null  float64
dtypes: float64(9), int64(1)
memory usage: 3.5 MB


In [69]:
import pandas as pd

df = pd.read_csv("CASP.csv")

# target = RMSD
y = df["RMSD"].values
X = df.drop(columns=["RMSD"]).values

print(X.shape, y.shape)

(45730, 9) (45730,)


In [70]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [71]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

linreg = LinearRegression()
linreg.fit(X_train, y_train)

preds = linreg.predict(X_train)
mse = mean_squared_error(y_train, preds)
print("Linear Regression Training MSE: ", mse)

lin_preds = linreg.predict(X_test)
mse = mean_squared_error(y_test, lin_preds)

print("Linear Regression Test MSE:", mse)

Linear Regression Training MSE:  26.721284736621268
Linear Regression Test MSE: 27.46102520966377


In [72]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

preds = rf.predict(X_train)
mse = mean_squared_error(y_train, preds)
print("RF Training MSE: ", mse)


rf_preds = rf.predict(X_test)
mse = mean_squared_error(y_test, rf_preds)

print("RF Test MSE:", mse)

RF Training MSE:  1.7375710577214643
RF Test MSE: 12.565003996491475


In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

True
NVIDIA RTX 1000 Ada Generation Laptop GPU
cuda


In [ ]:
import torch
import torch.nn as nn

class RegressionMLP(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, drop_out):
        super(RegressionMLP, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)  # input layer with width = length of the feature set
        # self.bn1 = nn.BatchNorm1d(hidden_size)
        self.dropout1 = nn.Dropout(p=drop_out) # only parts of the neurons are used.

        self.fc2 = nn.Linear(hidden_size, int(2*hidden_size))  # one hidden layer, try to add another one
        # self.bn2 = nn.BatchNorm1d(hidden_size)
        self.dropout2 = nn.Dropout(p=drop_out)

        self.fc3 = nn.Linear(int(2*hidden_size), int(2*hidden_size))  # one hidden layer, try to add another one
        # self.bn2 = nn.BatchNorm1d(hidden_size)
        self.dropout3 = nn.Dropout(p=drop_out)

        self.fc4 = nn.Linear(int(2*hidden_size), hidden_size)  # one hidden layer, try to add another one
        self.dropout4 = nn.Dropout(p=drop_out)

        self.fc5 = nn.Linear(hidden_size, int(hidden_size/2))  # one hidden layer, try to add another one
        self.dropout5 = nn.Dropout(p=drop_out)

        self.fc6 = nn.Linear(int(hidden_size/2), output_size)  # output layer
    
    # Define how the data flows through the layers
    def forward(self, x):            # forward pass
        x = torch.relu(self.fc1(x))  # ReLU activation function for input layer
        x = self.dropout1(x)

        x = torch.relu(self.fc2(x))  # ReLU activation function for hidden layer
        x = self.dropout2(x)

        x = torch.relu(self.fc3(x))  # ReLU activation function for hidden layer
        x = self.dropout3(x)
        
        x = torch.relu(self.fc4(x))  # ReLU activation function for hidden layer
        x = self.dropout4(x)

        x = torch.relu(self.fc5(x))  # ReLU activation function for hidden layer
        x = self.dropout5(x)

        x = self.fc6(x)   # no activation function for the output layer (because it is a regresion model!)
        return x

In [124]:
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)

X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

In [148]:
from torch.utils.data import TensorDataset, DataLoader
train_ds = TensorDataset(X_train_t, y_train_t)
test_ds  = TensorDataset(X_test_t, y_test_t)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=32)

In [157]:
# Hyperparameters
input_size = X_train.shape[1]
hidden_size = 256
output_size = 1

drop_out=0.3 # specifies how much of the NN remains randomly inactive
learning_rate = 0.0001
weight_decay = 0.0001

# Number of training iterations
num_epochs = 200

In [158]:
model = RegressionMLP(input_size, hidden_size, output_size, drop_out).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
criterion = nn.MSELoss()

In [159]:

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    total_samples = 0

    for data, targets in train_loader:
        data = data.to(device)
        targets = targets.to(device)

        outputs = model(data.to(torch.float32))

        loss = criterion(outputs, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # accumulate loss over batches
        batch_size = data.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

    avg_loss = total_loss / total_samples

    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {avg_loss:.4f}')

Epoch [10/200], Loss: 23.0030
Epoch [20/200], Loss: 21.0563
Epoch [30/200], Loss: 19.5237
Epoch [40/200], Loss: 18.3705
Epoch [50/200], Loss: 17.3479
Epoch [60/200], Loss: 16.5581
Epoch [70/200], Loss: 15.8046
Epoch [80/200], Loss: 15.2594
Epoch [90/200], Loss: 14.8894
Epoch [100/200], Loss: 14.2348
Epoch [110/200], Loss: 13.7583
Epoch [120/200], Loss: 13.4676
Epoch [130/200], Loss: 13.1085
Epoch [140/200], Loss: 12.8846
Epoch [150/200], Loss: 12.6187
Epoch [160/200], Loss: 12.3155
Epoch [170/200], Loss: 12.1292
Epoch [180/200], Loss: 11.7270
Epoch [190/200], Loss: 11.6177
Epoch [200/200], Loss: 11.2935


In [160]:
# Evaluate the model
model.eval()  # set model to eval mode (e.g. no dropout)

all_preds = []
all_targets = []

total_samples = 0
total_loss = 0

criterion = nn.MSELoss()

with torch.no_grad():
    for data, targets in test_loader:
        # data = data
        # targets = targets
        data = data.to(device)
        targets = targets.to(device)

        outputs = model(data.to(torch.float32))
        loss = criterion(outputs, targets)

        # accumulate loss over batches
        batch_size = data.size(0)
        total_loss += loss.item() * batch_size # multiply by batch size to 
        total_samples += targets.size(0)

        all_preds.extend(outputs.cpu().tolist()) # cpu() moves the tensor to cpu (if on gpu), tolist converts it to python list
        all_targets.extend(targets.cpu().tolist())

avg_loss = total_loss / total_samples

print(f"Average test Loss: {avg_loss:.4f}")


Average test Loss: 12.0737


#### Discussion points
1) Discuss your choice of model class.
2) How did you optimise your model? How did the best model perform?
3) How much time was needed for training the model and evaluations (approximation is enough)?
4) What limitations or shortcomings did you identify? What would be ideas to remedy or circumvent them?
5) In all its abstraction, what do the predictions of your model tell you?